In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error


# 1. Loading the Engineered Data & Spliting

df = pd.read_csv('../data/processed/train_FD001_engineered.csv')


train_engines = [i for i in range(1, 81)] # Training set
val_engines = [i for i in range(81, 101)] # Validation set

train_data = df[df['unit_nr'].isin(train_engines)]
val_data = df[df['unit_nr'].isin(val_engines)]


# 2. Defining Features (X) and Target (y)

cols_to_drop = ['unit_nr', 'time_cycles', 'RUL', 'RUL_clipped']

X_train = train_data.drop(columns=cols_to_drop)
y_train = train_data['RUL_clipped'] 

X_val = val_data.drop(columns=cols_to_drop)
y_val = val_data['RUL_clipped']


# 3. Training the Models
print("Training models... this might take a minute...")

# A. Ridge Regression (Linear Baseline)
ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)

# B. Random Forest (Bagging Baseline)
rf = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

# C. Gradient Boosting (Standard Boosting)
gbr = GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=42)
gbr.fit(X_train, y_train)

# D. eXtreme Gradient Boosting (Advanced Boosting)
xgb = XGBRegressor(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42, n_jobs=-1)
xgb.fit(X_train, y_train)


# 4. Evaluating each models (RMSE)

models = {'Ridge': ridge, 'Random Forest': rf, 'Gradient Boosting': gbr, 'XGBoost': xgb}
results = {}

print("\n--- Baseline Performance (RMSE - Lower is Better) ---")
for name, model in models.items():
    preds = model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, preds))
    results[name] = rmse
    print(f"{name.ljust(18)}: {rmse:.2f} cycles")

Training models... this might take a minute...

--- Baseline Performance (RMSE - Lower is Better) ---
Ridge             : 20.88 cycles
Random Forest     : 18.88 cycles
Gradient Boosting : 18.89 cycles
XGBoost           : 19.04 cycles
